# Steam Charts — Player Count Time Series

Scrapes monthly player count data from [steamcharts.com](https://steamcharts.com) for a curated list of games and visualizes their trajectories over time.

Data source: Steam Charts publishes monthly average / peak concurrent players for every Steam game, going back to ~2012.

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
from io import StringIO
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=0.95)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120

## 1. Games to track
A curated list of Steam app IDs mixing "immortal" candidates with games expected to fade. Edit freely.

App IDs come from Steam store URLs: `store.steampowered.com/app/{APP_ID}/...`

In [ ]:
GAMES = {
    # --- Expected immortals ---
    730:     'Counter-Strike 2',
    570:     'Dota 2',
    578080:  'PUBG',
    252490:  'Rust',
    440:     'Team Fortress 2',
    431960:  'Wallpaper Engine',
    289070:  'Civilization VI',
    105600:  'Terraria',
    374320:  'Dark Souls III',
    236850:  'Europa Universalis IV',
    238960:  'Path of Exile',
    
    # --- AAA singleplayer (expect decay) ---
    1174180: 'Red Dead Redemption 2',
    292030:  'The Witcher 3',
    489830:  'Skyrim Special Edition',
    1091500: 'Cyberpunk 2077',
    1245620: 'Elden Ring',
    
    # --- Recent releases (test the trajectory) ---
    2379780: 'Balatro',
    1966720: 'Lethal Company',
    1086940: 'Baldur\'s Gate 3',
    
    # --- Indie long-tail ---
    413150:  'Stardew Valley',
    367520:  'Hollow Knight',
    275850:  'No Man\'s Sky',
}

print(f"Tracking {len(GAMES)} games")

## 2. Scrape Steam Charts
`steamcharts.com/app/{id}` returns an HTML page with a monthly table. We use `pandas.read_html` to parse it directly — no BeautifulSoup needed.

In [ ]:
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (compatible; EDA-Notebook/1.0)'
}

def fetch_steamcharts(app_id, name):
    """Fetch monthly player count history for a single game."""
    url = f"https://steamcharts.com/app/{app_id}"
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        if r.status_code != 200:
            print(f"  ✗ {name}: HTTP {r.status_code}")
            return None
        # pandas.read_html finds all tables; Steam Charts has one main monthly table
        tables = pd.read_html(StringIO(r.text))
        if not tables:
            print(f"  ✗ {name}: no table found")
            return None
        df = tables[0].copy()
        df['app_id'] = app_id
        df['name'] = name
        return df
    except Exception as e:
        print(f"  ✗ {name}: {e}")
        return None

frames = []
for app_id, name in GAMES.items():
    print(f"  {name}...")
    df = fetch_steamcharts(app_id, name)
    if df is not None:
        frames.append(df)
    time.sleep(1.5)  # be polite

print(f"\nFetched {len(frames)} / {len(GAMES)} games")

## 3. Clean & combine

In [ ]:
raw = pd.concat(frames, ignore_index=True)
print("Raw columns:", raw.columns.tolist())
raw.head()

In [ ]:
# Steam Charts columns: Month | Avg. Players | Gain | % Gain | Peak Players
df = raw.rename(columns={
    'Month': 'month_str',
    'Avg. Players': 'avg_players',
    'Peak Players': 'peak_players'
})[['app_id', 'name', 'month_str', 'avg_players', 'peak_players']].copy()

# Drop the "Last 30 Days" row which has no parseable month
df = df[df['month_str'].str.contains(r'\d{4}', na=False)].copy()

# Parse month: "November 2024" → datetime
df['month'] = pd.to_datetime(df['month_str'], format='%B %Y', errors='coerce')
df = df.dropna(subset=['month'])

# Ensure numeric
df['avg_players'] = pd.to_numeric(df['avg_players'], errors='coerce')
df['peak_players'] = pd.to_numeric(df['peak_players'], errors='coerce')

df = df.sort_values(['name', 'month']).reset_index(drop=True)
print(f"Clean shape: {df.shape}")
print(f"Date range: {df['month'].min().date()} to {df['month'].max().date()}")
df.head()

## 4. Visualize — raw player counts
All games on one chart, log scale so both mega-hits and smaller games are visible.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

# Color by game, fade non-highlighted games
palette = sns.color_palette('tab20', n_colors=df['name'].nunique())

for (name, group), color in zip(df.groupby('name'), palette):
    ax.plot(group['month'], group['avg_players'], label=name, color=color, alpha=0.85, linewidth=1.4)

ax.set_yscale('log')
ax.set_ylabel('Average concurrent players (log scale)')
ax.set_xlabel('Month')
ax.set_title('Steam Charts — monthly average concurrent players')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## 5. Normalize to launch — the "decay" view
To compare survivability, plot each game as a percentage of its own peak, with x-axis set to "months since peak." Immortals stay flat, failures collapse.

In [ ]:
# Normalize: each game's players as % of that game's all-time peak
df['pct_of_peak'] = df.groupby('name')['avg_players'].transform(
    lambda x: (x / x.max()) * 100
)

# Months since peak (the reference point for each game)
def months_since_peak(group):
    peak_month = group.loc[group['avg_players'].idxmax(), 'month']
    group['months_since_peak'] = (
        (group['month'].dt.year - peak_month.year) * 12 +
        (group['month'].dt.month - peak_month.month)
    )
    return group

df = df.groupby('name', group_keys=False).apply(months_since_peak)

fig, ax = plt.subplots(figsize=(14, 7))

for (name, group), color in zip(df.groupby('name'), palette):
    # Only show post-peak trajectory
    post = group[group['months_since_peak'] >= 0].sort_values('months_since_peak')
    ax.plot(post['months_since_peak'], post['pct_of_peak'],
            label=name, color=color, alpha=0.85, linewidth=1.4)

ax.axhline(10, color='gray', linestyle='--', alpha=0.5, label='10% of peak')
ax.set_xlabel('Months since peak')
ax.set_ylabel('% of game\'s all-time peak')
ax.set_title('Post-peak decay — normalized by each game\'s own peak')
ax.set_ylim(0, 120)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## 6. The survivability verdict
Which games retained what % of their peak player base?

In [ ]:
# Latest month value as % of all-time peak
latest = (
    df.sort_values('month')
    .groupby('name')
    .agg(
        peak_players=('avg_players', 'max'),
        current_players=('avg_players', 'last'),
        months_of_data=('month', 'count'),
        first_month=('month', 'min'),
    )
)
latest['survivability_pct'] = (latest['current_players'] / latest['peak_players'] * 100).round(1)
latest = latest.sort_values('survivability_pct', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#c0392b' if v < 10 else '#e67e22' if v < 30 else '#27ae60' for v in latest['survivability_pct']]
ax.barh(latest.index, latest['survivability_pct'], color=colors)
ax.set_xlabel('Current players as % of all-time peak')
ax.set_title('Survivability ranking — green >30%, orange >10%, red <10%')
ax.axvline(10, color='gray', linestyle='--', alpha=0.5)
ax.axvline(30, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

latest[['peak_players', 'current_players', 'survivability_pct']].round(0)

## 7. Export

In [ ]:
df.to_csv('steamcharts_timeseries.csv', index=False)
latest.to_csv('steamcharts_survivability.csv')
print(f"Exported {len(df)} monthly records across {df['name'].nunique()} games")